### RAG Pipeline  
Data Ingestion to Vector DB Pipeline the workflow will be developed in below manner:
1-Read user defined documents using LangChain
2-Convert those documents to docs and metadata strings
3-Divide the content and metadata to chunks using Langchain Text Splitter
4-Convert Text chunks to Embeddings using all-MiniLM-L6-v2
5-Store the vectors into Vevtor Db Which is chroma
6-Then develop query analyzer to take user input then convert into embeddings
7-Run a match with vector DB data using SKLearn Cosine Similarity
8-Return results in form of list with data metrics
9-Augment user text for LLMs by adding context from vector DB and then adding context with query for llm to analyze 

In [ ]:
import os
import langchain
from langchain_community.document_loaders import PyPDFLoader,PyPDFium2Loader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [ ]:
#read all the pdf inside all the directory

def process_all_pdf(pdf_directory):
    """process all the pdfs"""
    all_documents=[]
    pdf_dir=Path(pdf_directory)

    #find all the files in loop
    pdf_files=list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files process")
    
    for pdf in pdf_files:
        print (f"processing of : {pdf.name}")
        try:
            loader=PyPDFLoader(str(pdf))
            documents=loader.load()
            #add metdata 
            for doc in documents:
                doc.metadata['source_file']=pdf.name
                doc.metadata['file_type']='pdf'

            all_documents.extend(documents)
            print (f" loaded {len(documents)} pages")
        except Exception as e:
            print (f"Error:{e}")
        print(f"\nTotal documents loaded : {len(all_documents)}")
    return all_documents


all_pdfs=process_all_pdf("../data/pdf")


Ignoring wrong pointing object 8 0 (offset 0)


Found 2 PDF files process
processing of : ISBAR-toolkit.pdf
 loaded 11 pages

Total documents loaded : 11
processing of : sample-local-pdf.pdf
 loaded 3 pages

Total documents loaded : 14


In [ ]:
all_pdfs

[Document(metadata={'producer': 'Acrobat Elements 7.0.5 (Windows)', 'creator': 'PScript5.dll Version 5.2.2', 'creationdate': '2010-04-11T14:16:03+10:00', 'author': 'kaneta', 'moddate': '2010-04-11T14:16:03+10:00', 'title': 'Microsoft Word - Hunter New England Health ISBAR toolkit text Final.doc', 'source': '..\\data\\pdf\\ISBAR-toolkit.pdf', 'total_pages': 11, 'page': 0, 'page_label': '1', 'source_file': 'ISBAR-toolkit.pdf', 'file_type': 'pdf'}, page_content='ISBAR revisited:  Identifying and Solving BARriers to \neffective clinical handover  \n \nProject toolkit \n \nClinical Governance \nHunter New England Health   \nMay 2009 \n 1 \n \n \nSection 1:  Introduction to ISBAR \n \nThis toolkit introduces the “ISBAR” framework for effective \ncommunication. It aims to give users the capacity to adapt, \nimplement and evaluate an approach to clinical communication \naround clinical handover in a health care setting or organisation.   \n \nThe toolkit is set out in three sections.  \n \nThe

In [ ]:
###chunking of documents using text splitting 

def split_documents(documents ,chunk_size=1000,chunk_overlap=200):
    """ split documents into smaller chunks for better RAG Performance"""
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ",""])
    split_documents=text_splitter.split_documents(documents)
    print(f"split{len(documents)} documents into {len(split_documents)} chunks")
    #show a chunking eg
    if split_documents:
        print(f"\n Example Chunks")
        print(f"Content :{split_documents[0].page_content[:200]}....") 
        print(f"Metadata :{split_documents[0].metadata}....")
    return split_documents
chunks=split_documents(all_pdfs)
chunks

split14 documents into 43 chunks

 Example Chunks
Content :ISBAR revisited:  Identifying and Solving BARriers to 
effective clinical handover  
 
Project toolkit 
 
Clinical Governance 
Hunter New England Health   
May 2009 
 1 
 
 
Section 1:  Introduction t....
Metadata :{'producer': 'Acrobat Elements 7.0.5 (Windows)', 'creator': 'PScript5.dll Version 5.2.2', 'creationdate': '2010-04-11T14:16:03+10:00', 'author': 'kaneta', 'moddate': '2010-04-11T14:16:03+10:00', 'title': 'Microsoft Word - Hunter New England Health ISBAR toolkit text Final.doc', 'source': '..\\data\\pdf\\ISBAR-toolkit.pdf', 'total_pages': 11, 'page': 0, 'page_label': '1', 'source_file': 'ISBAR-toolkit.pdf', 'file_type': 'pdf'}....


[Document(metadata={'producer': 'Acrobat Elements 7.0.5 (Windows)', 'creator': 'PScript5.dll Version 5.2.2', 'creationdate': '2010-04-11T14:16:03+10:00', 'author': 'kaneta', 'moddate': '2010-04-11T14:16:03+10:00', 'title': 'Microsoft Word - Hunter New England Health ISBAR toolkit text Final.doc', 'source': '..\\data\\pdf\\ISBAR-toolkit.pdf', 'total_pages': 11, 'page': 0, 'page_label': '1', 'source_file': 'ISBAR-toolkit.pdf', 'file_type': 'pdf'}, page_content='ISBAR revisited:  Identifying and Solving BARriers to \neffective clinical handover  \n \nProject toolkit \n \nClinical Governance \nHunter New England Health   \nMay 2009 \n 1 \n \n \nSection 1:  Introduction to ISBAR \n \nThis toolkit introduces the “ISBAR” framework for effective \ncommunication. It aims to give users the capacity to adapt, \nimplement and evaluate an approach to clinical communication \naround clinical handover in a health care setting or organisation.   \n \nThe toolkit is set out in three sections.  \n \nThe

### Embedding 
embedinng of the chunks .We will use a Hugging face sentecnce transformer model ,this includes converting text into vectors

In [72]:


import numpy as np
from sentence_transformers import SentenceTransformer # embedding model will be avilale inside this
import uuid #everyrecord inserted to db will have an id will use this library
from uuid import uuid4
import chromadb # for vector db
from chromadb.config import Settings
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity # apply cosine for retrieval of data


class EmbeddingManager:
    """Handles document embedding generation using sentence transformer"""
    def __init__(self,model_name:str="all-MiniLM-L6-v2"): ###It maps sentences & paragraphs to a 384 dimensional dense vector space
     """Initialize the embedding manager
     Args:
         model_name:HuggingFace Model name for sentence embeddings
     """
     self.model_name=model_name
     self.model=None
     self._load_model()
    
    def _load_model(self): ##protected function available within this class only
       """Load the sentence transformer model"""
       try:
          print(f"Loading Embedding model: {self.model_name}")
          self.model=SentenceTransformer(self.model_name)
          print(f"Model Loaded Successfully.Embedding Dimension : {self.model.get_sentence_embedding_dimension()}")
       except Exception as e:
            print(f"Loading Embedding model: {self.model_name}:{e}")
            raise
    def generate_embeddings(self,texts:List[str]) -> np.ndarray: ###generate a lis of texts and convert it into numpy array
       """Generate Embeddings for a list of texts 
          Args:
               texts:List of text strings to embded
          Returns:
               numpy array of embeddings with shape (len(texts),embedding_dim)
       """
       if self.model is None:
          raise ValueError("Model not Loaded")
       print(f"Generate Embeddings for {len(texts)}texts....")
       embeddings=self.model.encode(texts,show_progress_bar=True)
       print(f"Generate Embeddings with shape {embeddings.shape}..")
       return embeddings
    
    #def get_sentence_embedding_dimesion(self):
     # """get the embedding dimension of the model"""
     # if not self.model:
       #  raise ValueError("model not loaded")
     # return self.model.get_sentence_embedding_dimesion()

######Initilize the embedding manager
embedding_manager=EmbeddingManager()
embedding_manager

Loading Embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11445.58it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model Loaded Successfully.Embedding Dimension : 384


C:\Users\VAR students\AppData\Local\Temp\ipykernel_26176\3317241548.py:27: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model Loaded Successfully.Embedding Dimension : {self.model.get_sentence_embedding_dimension()}")


### Vector DB

In [73]:
class VectorStore:
    """Handles document embedding generation using sentence transformer"""
    def __init__(self,collection_name:str="pdf_documents",persist_directory:str ="../data/vector_store"): ###It maps sentences & paragraphs to a 384 dimensional dense vector space
     """Initialize the vector store
     Args:
         collection_name:collection_name
         persist_directory: Directory to persist the vector store
     """
     self.collection_name=collection_name
     self.persist_directory=persist_directory
     self.client=None
     self.collection=None
     self.initialize_store() ## to initialize the vector store

    def initialize_store(self):
       """Initilize the chromaDB Client and Collection"""
       try:
          ##Create peristent ChromaDB client 
          os.makedirs(self.persist_directory,exist_ok=True)
          self.client=chromadb.PersistentClient(path=self.persist_directory)
          
          ###create or GET collection
          self.collection=self.client.get_or_create_collection(
             name=self.collection_name,
             metadata={"description":"PDF document embeddings for RAG"}
          )
          print(f"Vector Store Initialized.Collection : {self.collection_name}")
          print(f"Existing Documkents in collection: {self.collection.count()}")
       except Exception as e:
            print(f"Error initialzing the vector store:{e}")
            raise
   ### adding the documents 
    def add_documents (self ,documents:List[Any],embeddings:np.ndarray):
       """Add documents and their embeddings to the vector store
          Args :
            documents :List of lnagchain documents
            embeddings: corresponding embeddings for the documents
       """
       if len(documents) != len(embeddings):
          raise ValueError ("Number of docs must match number of embeddings")
          
       print(f"Adding {len(documents)} documents to the vetor store...")

       ###prepare data for chromaDB
       ids=[]
       metadatas=[]
       documents_text=[]
       embeddings_list=[]

       for i, (doc,embedding) in enumerate(zip(documents,embeddings)):
          ##generate a unique id
          doc_id=f"doc_{uuid.uuid4().hex[:8]}_{i}" ## id for a specific record
          ids.append(doc_id)

          ###prepare metadata
          metadata=dict(doc.metadata)
          metadata['doc_index']=i
          metadata['content_length']=len(doc.page_content)
          metadatas.append(metadata)

          #document content
          documents_text.append(doc.page_content)
          
          #embedding
          embeddings_list.append(embedding.tolist())
          
          #add to collection
          try:
             self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
             )
             print(f"sucessfully added  {len(documents)} documents to vector store")
             print(f"Total documents in collection:{self.collection.count()}")
          except  Exception as e:
             print(f"Error adding documens to vector store:{e}")
             raise 

vectorStore=VectorStore()

vectorStore
           
       

Vector Store Initialized.Collection : pdf_documents
Existing Documkents in collection: 107


In [74]:
### convert the text into embeddings

texts=[doc.page_content for doc in chunks]

## generate the embeddings
embeddings=embedding_manager.generate_embeddings(texts)

####store in the vetor database
vectorStore.add_documents(chunks,embeddings)

Generate Embeddings for 43texts....


Batches: 100%|██████████| 2/2 [00:00<00:00,  5.05it/s]


Generate Embeddings with shape (43, 384)..
Adding 43 documents to the vetor store...
sucessfully added  43 documents to vector store
Total documents in collection:108
sucessfully added  43 documents to vector store
Total documents in collection:109
sucessfully added  43 documents to vector store
Total documents in collection:110
sucessfully added  43 documents to vector store
Total documents in collection:111
sucessfully added  43 documents to vector store
Total documents in collection:112
sucessfully added  43 documents to vector store
Total documents in collection:113
sucessfully added  43 documents to vector store
Total documents in collection:114
sucessfully added  43 documents to vector store
Total documents in collection:115
sucessfully added  43 documents to vector store
Total documents in collection:116
sucessfully added  43 documents to vector store
Total documents in collection:117
sucessfully added  43 documents to vector store
Total documents in collection:118
sucessfully a

##### RAG Retirever pipeline

In [94]:
class RagRetriever:
    """Handles query based retreivel from the vector store"""
    def __init__(self,vector_store:VectorStore,embedding_manager:EmbeddingManager): ###It maps sentences & paragraphs to a 384 dimensional dense vector space
     """Initialize the retiever
     Args:
         vector_store:vector store containing the document embeddings
         embedding_manager: Manager for generating the embeddings
     """
     self.vector_store=vector_store
     self.embedding_manager=embedding_manager
    def retrieve(self,query:str,top_k=5,score_threshold:float=0.0) -> List[Dict[str,Any]]:
        """
        Retrieve relevant documents for a query
        Args :
           query: the search query
           top_k: Number of top results to return
           score_threshhold: minimum similarity score threshold

        Returns:
           List of dictuonaries containing retrieved documents metadata
        """
        print(f"Retrieveing documents for query  '{query}' ")
        print(f"Top K:{top_k} ,Score threshhold :{score_threshold}")
       
       ###first generate query into embedding 
        query_embedding= self.embedding_manager.generate_embeddings([query])[0]
        
        ##search in  vector store 
        try:
           results=self.vector_store.collection.query(
              query_embeddings=[query_embedding.tolist()],
              n_results=top_k
           )
           ##process the results
           retrieved_docs=[]
           if results['documents'] and results['documents'][0]:
              documents=results['documents'][0]
              metadatas=results['metadatas'][0]
              distances=results['distances'][0]
              ids=results['ids'][0]
              
              for i, (doc_id,document,metdata,distance) in enumerate(zip(ids,documents,metadatas,distances)):
                 #### convert distance to similarity score (ChromaDB uses cosine distance)
                 similarity_score=1-distance
                 if similarity_score>=score_threshold:
                    retrieved_docs.append({
                       'id':doc_id,
                       'content':document,
                       'metdata':metdata,
                       'similarity_score':similarity_score,
                       'distance':distance,
                       'rank':i+1
                    })
                    print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")     
                 else:
                    print("No documents found")
                 return retrieved_docs   
        except Exception as e:
           print(f"error during retrievel : {e}")
           return[]                                       
        
ragRetriever=RagRetriever(vectorStore,embedding_manager)
ragRetriever

In [95]:
ragRetriever.retrieve("what is ISBAR")

Retrieveing documents for query  'what is ISBAR' 
Top K:5 ,Score threshhold :0.0
Generate Embeddings for 1texts....


Batches: 100%|██████████| 1/1 [00:00<00:00, 117.54it/s]

Generate Embeddings with shape (1, 384)..
Retrieved 1 documents (after filtering)


[{'id': 'doc_cb8c1bb2_7',
  'content': '4\n \nFrequently asked questions about ISBAR \nWhat is ISBAR? \nThe ISBAR framework represents a standardised approach to \ncommunication which can be used in any situation. It stands for \nIntroduction, Situation, Background, Assessment and \nRecommendation. \n \nWhat are the advantages of ISBAR? \nThe ISBAR framework  \n• Ensures completeness of information and reduces \nlikelihood of missed data \n\x83 Is an easy and focussed way to set expectations for what \nwill be communicated \n\x83 Ensures a recommendation is clear and professional \n\x83 Gives confidence in communication \n\x83 focuses not on the people who are communicating but on \nthe problem itself. \n \nWhy would I use ISBAR?   \nKey reasons for using ISBAR are  \n\x83 It is portable, memorable and easy to use \n\x83 Can be used to present information clearly in any situation \n\x83 Helps you to organise what you’re going to say \n\x83 Standardises communication between everyone  \

### Integration of VectorBD Context Pipleline with LLM output

In [110]:
### simple RAG pipelie with GROK LLM
from langchain_groq import ChatGroq ### provides a Language Processing Unit (LPU) inference engine that offers incredibly high-speed inference for large language models
import os
from dotenv import load_dotenv ##  file is specifically used to store all the variables like API keys 
load_dotenv()

## initialize the grok llm using API key from .env
grok_api_key=os.getenv("GROQ_API_KEY")

llm=ChatGroq(api_key=grok_api_key,model_name="llama-3.1-8b-instant",temperature=0.1,max_tokens=1024)

### Simple RAG Function from retrive context from generate response

def rag_simple(query,retreiever,llm,top_k=3):
    ## retieve the context
    results=retreiever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question"
    ### generate the answer using GROQ LLM
    prompt=f"""Use the following context to answer the question precisely
    context :
    {context}
    Question : {query}
    Answer:
    """
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

answer=rag_simple("what is ISBAR?",ragRetriever,llm)
print(answer)

Retrieveing documents for query  'what is ISBAR?' 
Top K:3 ,Score threshhold :0.0
Generate Embeddings for 1texts....


Batches: 100%|██████████| 1/1 [00:00<00:00, 250.03it/s]

Generate Embeddings with shape (1, 384)..
Retrieved 1 documents (after filtering)
ISBAR stands for Introduction, Situation, Background, Assessment, and Recommendation. It is a standardized approach to communication that can be used in any situation.


In [121]:
### RAG With more features
def rag_advanced(query,retreiever,llm,top_k=5,min_score=0.2,return_context=False):

    """RAG pipeline with more features :
       Returns answers ,sources,confidence score ,and null context optionally
    
    """
    ## retieve the context
    results=retreiever.retrieve(query,top_k=top_k,score_threshold=min_score)
    if not results:
       return {
            'answer': "No relevant context found.",
            'sources': [],
            'confidence': 0.0,
            'context': "" if return_context else None
}
    
    ### prepare context and sources
    context=context = "\n\n".join([doc.get('content', '') for doc in results])
    sources=[]
    for doc in results:
        metadata=doc.get('metadata',{})
        sources.append({
                    'sources':metadata.get('source_file',metadata.get('source','unknown')),
                    'page':metadata.get('page','unknown'),
                    'score':doc.get('similarity_score',0.0),
                    'preview':doc.get('content','')[:300]+'......',

        })
    #sources=[{
      #  'sources':doc['metadata'].get('source_file',doc['metadata'].get('source','unknown')),
      #  'page':doc['metadata'].get('page','unknown'),
     #   'score':doc['similarity_score'],
     #   'preview':doc['content'][:300]+'......',
   # } for doc in results]
    confidence=max([doc.get('similarity_score',0.0) for doc in results])
##genrate the response/answer
    prompt=f"""Use the following context to answer the question precisely.\n{context}\n\nQuestion:{query}\n\nAnswer """
   
    response=llm.invoke([prompt.format(context=context,query=query)])
    output={
     'answer':response.content,
     'sources':sources,
     'confidence': confidence
    }
    if return_context:
        output['context']=context
    return output

####user prompt
result=rag_advanced("what is isbar",ragRetriever,llm,top_k=3,min_score=0.1,return_context=True)
print("Answer:",result['answer'])
print("Sources:",result['sources'])
print("confidence:",result['confidence'])
print("context preview:",result['context'][:300])

Retrieveing documents for query  'what is isbar' 
Top K:3 ,Score threshhold :0.1
Generate Embeddings for 1texts....


Batches: 100%|██████████| 1/1 [00:00<00:00, 158.47it/s]

Generate Embeddings with shape (1, 384)..
Retrieved 1 documents (after filtering)


Answer: ISBAR is a standardized approach to communication that stands for:

1. Introduction
2. Situation
3. Background
4. Assessment
5. Recommendation

It is a framework that can be used in any situation to ensure completeness of information, clarity, and professionalism in communication.
Sources: [{'sources': 'unknown', 'page': 'unknown', 'score': 0.543599396944046, 'preview': '4\n \nFrequently asked questions about ISBAR \nWhat is ISBAR? \nThe ISBAR framework represents a standardised approach to \ncommunication which can be used in any situation. It stands for \nIntroduction, Situation, Background, Assessment and \nRecommendation. \n \nWhat are the advantages of ISBAR? \nThe ISBAR......'}]
confidence: 0.543599396944046
context preview: 4
 
Frequently asked questions about ISBAR 
What is ISBAR? 
The ISBAR framework represents a standardised approach to 
communication which can be used in any situation. It stands for 
Introduction, Situation, Background, Assessment and 
Recommendation